<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EA%B0%9C%EB%A1%A0_12%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## I. 단답형 문제

[단답형 1]

훈련 데이터(train data)에 대해서는 정확도가 높게 나오지만, 검증 데이터(validation data)에 대해서는 정확도가 낮게 나오는 현상을 무엇이라고 하며, 이 현상이 발생하기 시작하는 시점(Epoch)을 학습 곡선(Loss curve)에서 어떻게 판단할 수 있습니까?

답:

1. 현상: 과학습 (Overfitting)
2. 판단 시점: 검증 데이터 손실이 훈련 손실보다 커지는 시점, 즉 손실이 더이상 감소하지 않고 증가하기 시작하는 시점

[단답형 2]

12차시 강의안에서 배운 튜닝 기법 중, 훈련(training) 시와 평가(evaluation) 시에 다르게 동작해야 해서 `model.train()`과 `model.eval()` 모드 전환이 반드시 필요한 기법 2가지를 쓰시오.

답:

1. 드롭아웃 : train()에서는 지정된 비율만큼 뉴런을 무작위로 비활성화 시켜 과학습을 방지하고, eval()모드에서는 모든 뉴런을 사용하여 일관된 예측을 수행한다.
2. 배치 정규화 : train()에서는 현재  미니배치의 평균/분산으로 정규화하고 이동 평균을 업데이트 하고, eval()모드에서는 훈련중에 축적된 이동 평균을 사용하여 데이터를 일관되게 정규화 한다.

---

## II. 코드 문제

**(아래 문제들은 12차시 실습 코드(.ipynb)의 `CNN_v2`, `CNN_v3`, `CNN_v4` 클래스 및 `optim`, `transforms` 관련 내용을 참고하여 푸시오.)**

[코드 1]

`CNN_v2` 모델의 최적화 함수(optimizer)를 `optim.SGD`에서 `optim.Adam`으로 변경하는 코드를 작성하십시오. (learning rate는 0.01로 동일하게 설정)

In [ ]:
# 12차시 코드 셀의 optim, net 변수가 이미 선언되어 있다고 가정합니다.
# net = CNN_v2(n_output).to(device)
# lr = 0.01

# TODO: 최적화 함수를 optim.Adam으로 변경하시오.
optimizer = optim.Adam(net.parameters(), lr = 0.01)

[코드 2]

`CNN_v2` 모델의 분류기(classifier) 부분에서, 첫 번째 선형 레이어(`nn.Linear(2048, 128)`)와 활성화 함수(`self.relu`) *다음에* 25%의 비율로 `nn.Dropout` 레이어를 추가하여 `CNN_v3` 모델을 완성하십시오.

In [ ]:
# 12차시 코드 셀의 nn, CNN_v2, self.relu 등이 이미 선언되어 있다고 가정합니다.

class CNN_v3(nn.Module):
    def __init__(self, n_output):
        super().__init__()
        # ... (features 부분은 CNN_v2와 동일하다고 가정) ...
        # self.features = ...
        # self.relu = nn.ReLU(inplace=True)

        self.classifier = nn.Sequential(
            nn.Linear(2048, 128),
            self.relu,
            # TODO: 여기에 25% 비율의 드롭아웃 레이어를 추가하시오.
            nn.Dropout(0.25)
            nn.Linear(128, n_output)
        )

    def forward(self, x):
        # ... (forward 함수는 CNN_v2와 동일하다고 가정) ...
        # x = self.features(x)
        # x = x.view(x.size(0), -1)
        # x = self.classifier(x)
        return x

[코드 3]

`CNN_v3` 모델의 `features` 부분에 배치 정규화(Batch Normalization)를 적용하고자 합니다. `conv1` (32채널) 다음, 그리고 활성화 함수(`self.relu`) *이전에* `nn.BatchNorm2d` 레이어를 추가하여 `CNN_v4` 모델의 일부를 완성하십시오.

In [ ]:
# 12차시 코드 셀의 nn이 이미 선언되어 있다고 가정합니다.

class CNN_v4(nn.Module):
    def __init__(self, n_output):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)

        # TODO: conv1 (32채널)을 위한 배치 정규화 레이어를 정의하시오.
        self.bn1 = nn.BatchNorm2d(32)

        self.relu = nn.ReLU(inplace=True)
        # ... (other layers defined here) ...

        self.features = nn.Sequential(
            self.conv1,
            # TODO: 정의한 배치 정규화 레이어를 여기에 적용하시오.
            self.bn1,
            self.relu,
            # ... (rest of features sequence) ...
        )
        # ... (classifier defined here) ...

    def forward(self, x):
        # ... (forward function) ...
        return x

[코드 4]

CIFAR10 훈련 데이터셋(train set)에 데이터 증강(Data Augmentation) 기법을 적용하려고 합니다. `transforms.Compose`를 사용하여, 50% 확률의 '좌우 반전(RandomHorizontalFlip)'을 `ToTensor` *이전에* 적용하고, 50% 확률의 '무작위 지우기(RandomErasing)'를 `Normalize` *이후에* 적용하는 `transform_train`을 완성하십시오. (Normalize는 `(0.5, 0.5)` 사용)

In [ ]:
# 12차시 코드 셀의 transforms가 이미 선언되어 있다고 가정합니다.

transform_train = transforms.Compose([
    # TODO: 50% 확률의 좌우 반전을 추가하시오.
    transforms.RandomHorizontalFlip(p=0.5)
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
    # TODO: 50% 확률의 무작위 지우기를 추가하시오. (기본 파라미터, p=0.5)
    transforms.RandomErasing(p=0.5)
])